# EfficientNet Transfer Learning — Kaggle Version
# ===================================================
## အဓိကအချက်များ
- **Dataset**: Kaggle ထဲမှ food-101 (သို့) ကိုယ်တိုင် upload လုပ်ထားသော dataset
- **Model**: EfficientNet B0–B7 / V2-S/M/L (PyTorch torchvision)
- **Strategy**: Phase 1 — Classifier only → Phase 2 — Full fine-tune
- **Output**: `/kaggle/working/saved_models/` ထဲသို့ weights + full model + plots + report

> **Kaggle Dataset Setup**: Notebook sidebar → **+ Add Data** → food-101 ကိုရှာပြီး add လုပ်ပါ။  
> Dataset path: `/kaggle/input/food-101/food-101/images/`

In [ ]:
# ── 1. Package check (Kaggle တွင် pre-installed ဖြစ်သောကြောင့် install မလိုပါ) ──
import subprocess, sys

def _check(pkg):
    try:
        __import__(pkg)
        print(f"  ✔ {pkg}")
    except ImportError:
        print(f"  ✘ {pkg} — installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["torch", "torchvision", "sklearn", "PIL", "matplotlib", "numpy"]:
    _check(pkg.replace("sklearn", "scikit-learn").replace("PIL", "Pillow"))
print("All packages ready.")

In [ ]:
# ── 2. Imports ──────────────────────────────────────────────────────────────────
import os
import shutil
import random
import time
from pathlib import Path
from itertools import cycle

import numpy as np
import matplotlib
matplotlib.use("Agg")          # GUI-less (notebook display 은 plt.show() 또는 IPython.display 사용)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image as IPImage

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize

print("Imports complete.")

## Configuration
`DATA_DIR` နှင့် `SELECTED_CLASSES` ကိုသာ ပြင်ဆင်ပြီး run ပါ။

In [ ]:
# ── 3. Configuration ────────────────────────────────────────────────────────────

# ① Dataset path — Kaggle food-101 dataset (sidebar → + Add Data → food-101)
DATA_DIR = Path("/kaggle/input/food-101/food-101/images")

# ② Class subset — None ဆိုရင် ကျန်တဲ့ class များကို ဆက်သုံးမည်; list ဖြင့် သတ်မှတ်ရင် subset သာ သုံးမည်
SELECTED_CLASSES = [
    "apple_pie",
    "chicken_wings",
    "club_sandwich",
    "donuts",
    "pizza",
    "sushi",
]  # None ဟုပြောင်းလျှင် full 101 classes

# ③ EfficientNet variant
# V1: "efficientnet_b0" ~ "efficientnet_b7"
# V2: "efficientnet_v2_s", "efficientnet_v2_m", "efficientnet_v2_l"
MODEL_NAME = "efficientnet_b0"

# ④ Hyperparameters
BATCH_SIZE              = 32
NUM_EPOCHS_CLASSIFIER   = 10
NUM_EPOCHS_FINETUNE     = 5
LR_CLASSIFIER           = 1e-3
LR_FINETUNE             = 1e-4
TRAIN_RATIO             = 0.70
VAL_RATIO               = 0.15
TEST_RATIO              = 0.15

# ⑤ ImageNet normalization
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

# ⑥ Kaggle T4/P100 GPU 사용 + Linux multi-worker
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2
PIN_MEMORY  = DEVICE.type == "cuda"

# ⑦ Save / output directory
SAVE_BASE = Path("/kaggle/working/saved_models")

print(f"Device     : {DEVICE}")
print(f"Model      : {MODEL_NAME}")
print(f"Data dir   : {DATA_DIR}")
print(f"Classes    : {SELECTED_CLASSES if SELECTED_CLASSES else 'ALL'}")
print(f"Save to    : {SAVE_BASE}")

In [ ]:
# ── 4. EfficientNet variant table ───────────────────────────────────────────────
EFFICIENTNET_VARIANTS = {
    "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1,  224),
    "efficientnet_b1": (models.efficientnet_b1, models.EfficientNet_B1_Weights.IMAGENET1K_V2,  240),
    "efficientnet_b2": (models.efficientnet_b2, models.EfficientNet_B2_Weights.IMAGENET1K_V1,  260),
    "efficientnet_b3": (models.efficientnet_b3, models.EfficientNet_B3_Weights.IMAGENET1K_V1,  300),
    "efficientnet_b4": (models.efficientnet_b4, models.EfficientNet_B4_Weights.IMAGENET1K_V1,  380),
    "efficientnet_b5": (models.efficientnet_b5, models.EfficientNet_B5_Weights.IMAGENET1K_V1,  456),
    "efficientnet_b6": (models.efficientnet_b6, models.EfficientNet_B6_Weights.IMAGENET1K_V1,  528),
    "efficientnet_b7": (models.efficientnet_b7, models.EfficientNet_B7_Weights.IMAGENET1K_V1,  600),
    "efficientnet_v2_s": (models.efficientnet_v2_s, models.EfficientNet_V2_S_Weights.IMAGENET1K_V1, 384),
    "efficientnet_v2_m": (models.efficientnet_v2_m, models.EfficientNet_V2_M_Weights.IMAGENET1K_V1, 480),
    "efficientnet_v2_l": (models.efficientnet_v2_l, models.EfficientNet_V2_L_Weights.IMAGENET1K_V1, 480),
}

def get_img_size(variant=MODEL_NAME):
    return EFFICIENTNET_VARIANTS[variant][2]

IMG_SIZE = get_img_size(MODEL_NAME)
print(f"Input image size: {IMG_SIZE}x{IMG_SIZE}")

## Dataset — Class Subset Copy & Split
Kaggle `/kaggle/input/` read-only ဖြစ်သောကြောင့် selected classes ကို `/kaggle/working/data/` ထဲသို့ copy လုပ်ပြီး train/val/test split လုပ်မည်။

In [ ]:
# ── 5. Prepare working dataset directory ────────────────────────────────────────

WORKING_DATA = Path("/kaggle/working/data/raw")   # selected classes

def prepare_classes(src_dir: Path, dest_dir: Path, classes):
    """
    src_dir/class_name/*.jpg → dest_dir/class_name/*.jpg  (symlink or copy)
    classes=None  →  전체 클래스 사용
    """
    if dest_dir.exists():
        print(f"Already prepared: {dest_dir}")
        return

    all_classes = sorted(d.name for d in src_dir.iterdir() if d.is_dir())
    selected    = classes if classes else all_classes
    missing     = [c for c in selected if c not in all_classes]
    if missing:
        raise ValueError(f"Classes not found in dataset: {missing}")

    valid_ext = {'.jpg', '.jpeg', '.png', '.bmp'}
    for cls in selected:
        (dest_dir / cls).mkdir(parents=True, exist_ok=True)
        for f in (src_dir / cls).iterdir():
            if f.suffix.lower() in valid_ext:
                shutil.copy2(f, dest_dir / cls / f.name)
        count = len(list((dest_dir / cls).iterdir()))
        print(f"  {cls}: {count} images copied")

prepare_classes(DATA_DIR, WORKING_DATA, SELECTED_CLASSES)
print(f"\nWorking data ready: {WORKING_DATA}")

In [ ]:
# ── 6. Auto train/val/test split ────────────────────────────────────────────────

SPLIT_DIR = Path("/kaggle/working/data/split")

def auto_split(src_dir: Path, dest_dir: Path,
               train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, seed=42):
    """src_dir/class/*.jpg  →  dest_dir/train|val|test/class/*.jpg"""
    if (dest_dir / "train").exists():
        print(f"Split already exists at {dest_dir} — skipping.")
        return

    random.seed(seed)
    valid_ext   = {'.jpg', '.jpeg', '.png', '.bmp'}
    class_names = sorted(d.name for d in src_dir.iterdir() if d.is_dir())

    for split in ("train", "val", "test"):
        for cls in class_names:
            (dest_dir / split / cls).mkdir(parents=True, exist_ok=True)

    for cls in class_names:
        files = [f for f in (src_dir / cls).iterdir() if f.suffix.lower() in valid_ext]
        random.shuffle(files)
        n       = len(files)
        n_train = int(n * train_ratio)
        n_val   = int(n * val_ratio)
        splits  = {
            "train": files[:n_train],
            "val":   files[n_train : n_train + n_val],
            "test":  files[n_train + n_val :],
        }
        for split_name, split_files in splits.items():
            for f in split_files:
                shutil.copy2(f, dest_dir / split_name / cls / f.name)
        print(f"  {cls}: train={len(splits['train'])} "
              f"val={len(splits['val'])} test={len(splits['test'])}")

auto_split(WORKING_DATA, SPLIT_DIR)
print(f"\nSplit ready: {SPLIT_DIR}")

In [ ]:
# ── 7. Dataset class + Transforms ───────────────────────────────────────────────

class ImageFolderDataset(Dataset):
    """root/class_name/image.jpg structure ကို load မည်"""
    def __init__(self, root_dir, transform=None):
        self.samples    = []
        self.transform  = transform
        self.class_names = sorted(
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        )
        self.class_to_idx = {c: i for i, c in enumerate(self.class_names)}

        valid_ext = ('.jpg', '.jpeg', '.png', '.bmp')
        for class_name in self.class_names:
            class_dir = os.path.join(root_dir, class_name)
            for fname in os.listdir(class_dir):
                if fname.lower().endswith(valid_ext):
                    self.samples.append((
                        os.path.join(class_dir, fname),
                        self.class_to_idx[class_name]
                    ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


def get_train_transform(img_size):
    return transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])


def get_test_transform(img_size):
    resize = int(img_size / 0.875)
    return transforms.Compose([
        transforms.Resize(resize),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])


print("Dataset class and transforms defined.")

In [ ]:
# ── 8. DataLoaders ──────────────────────────────────────────────────────────────

def create_dataloaders(train_dir, test_dir, img_size):
    train_full = ImageFolderDataset(train_dir, transform=get_train_transform(img_size))
    test_ds    = ImageFolderDataset(test_dir,  transform=get_test_transform(img_size))

    val_size   = int(len(train_full) * VAL_RATIO)
    train_size = len(train_full) - val_size
    train_sub, val_sub = random_split(
        train_full, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    lkw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
               pin_memory=PIN_MEMORY, persistent_workers=(NUM_WORKERS > 0))

    train_loader = DataLoader(train_sub, shuffle=True,  **lkw)
    val_loader   = DataLoader(val_sub,   shuffle=False, **lkw)
    test_loader  = DataLoader(test_ds,   shuffle=False, **lkw)

    print(f"Classes    : {train_full.class_names}")
    print(f"Train      : {train_size} | Val: {val_size} | Test: {len(test_ds)}")
    return train_loader, val_loader, test_loader, len(train_full.class_names), train_full.class_names


train_loader, val_loader, test_loader, NUM_CLASSES, CLASS_NAMES = \
    create_dataloaders(SPLIT_DIR / "train", SPLIT_DIR / "test", IMG_SIZE)

In [ ]:
# ── 9. Sample images preview ────────────────────────────────────────────────────

def show_samples(split_dir: Path, class_names, n_per_class=4):
    fig, axes = plt.subplots(len(class_names), n_per_class,
                             figsize=(n_per_class * 3, len(class_names) * 3))
    if len(class_names) == 1:
        axes = [axes]
    for row, cls in enumerate(class_names):
        cls_dir = split_dir / "train" / cls
        imgs    = random.sample(list(cls_dir.iterdir()), min(n_per_class, len(list(cls_dir.iterdir()))))
        for col, img_path in enumerate(imgs):
            ax = axes[row][col]
            ax.imshow(mpimg.imread(img_path))
            ax.axis("off")
            if col == 0:
                ax.set_title(cls, fontsize=9, loc="left")
    plt.suptitle("Sample images per class", fontsize=12)
    plt.tight_layout()
    plt.savefig("/kaggle/working/sample_images.png", dpi=100)
    plt.show()
    print("Sample images saved to /kaggle/working/sample_images.png")

show_samples(SPLIT_DIR, CLASS_NAMES)

## Model — EfficientNet + Custom Classifier Head

In [ ]:
# ── 10. Build model ─────────────────────────────────────────────────────────────

def create_model(num_classes, variant=MODEL_NAME):
    model_fn, weights, _ = EFFICIENTNET_VARIANTS[variant]
    model = model_fn(weights=weights)

    # Backbone ကို freeze
    for param in model.parameters():
        param.requires_grad = False

    # Custom classifier head (Dropout → Linear 512 → ReLU → Dropout → Linear num_classes)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(512, num_classes),
    )

    model = model.to(DEVICE)
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{variant}]")
    print(f"  Total params    : {total:,}")
    print(f"  Trainable params: {trainable:,}  (frozen: {total - trainable:,})")
    return model


model = create_model(NUM_CLASSES, MODEL_NAME)

## Training

### Phase 1 — Classifier Only (backbone frozen)
### Phase 2 — Full Fine-tune (all layers unfrozen)

In [ ]:
# ── 11. Train / evaluate helpers ────────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += labels.size(0)
    return running_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            correct      += (outputs.argmax(1) == labels).sum().item()
            total        += labels.size(0)
    return running_loss / total, correct / total


def train_loop(model, train_loader, val_loader, criterion, optimizer,
               scheduler, num_epochs, phase_name):
    history      = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_state   = None

    print(f"\n{'='*65}")
    print(f" {phase_name}")
    print(f"{'='*65}")

    for epoch in range(num_epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        scheduler.step(vl_loss)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(vl_loss)
        history["val_acc"].append(vl_acc)

        marker = ""
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            marker       = "  ✓ best"

        print(f"  Epoch [{epoch+1:2d}/{num_epochs}]  "
              f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  |  "
              f"val_loss={vl_loss:.4f}  val_acc={vl_acc:.4f}  "
              f"({time.time()-t0:.1f}s){marker}")

    if best_state:
        model.load_state_dict(best_state)
    print(f"\n  Best Val Acc: {best_val_acc:.4f}")
    return history


print("Training helpers defined.")

In [ ]:
# ── 12. Phase 1: Classifier Only ────────────────────────────────────────────────

criterion_p1  = nn.CrossEntropyLoss()
optimizer_p1  = optim.Adam(model.classifier.parameters(), lr=LR_CLASSIFIER)
scheduler_p1  = optim.lr_scheduler.ReduceLROnPlateau(optimizer_p1, factor=0.5, patience=2)

history_p1 = train_loop(
    model, train_loader, val_loader,
    criterion_p1, optimizer_p1, scheduler_p1,
    NUM_EPOCHS_CLASSIFIER,
    "Phase 1: Classifier Only (backbone frozen)"
)

In [ ]:
# ── 13. Phase 2: Full Fine-tune ─────────────────────────────────────────────────

# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"All layers unfrozen — trainable params: {trainable:,}")

criterion_p2 = nn.CrossEntropyLoss()
optimizer_p2 = optim.Adam([
    {"params": [p for n, p in model.named_parameters() if "classifier" not in n],
     "lr": LR_FINETUNE / 10},
    {"params": model.classifier.parameters(), "lr": LR_FINETUNE},
])
scheduler_p2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_p2, factor=0.5, patience=2)

history_p2 = train_loop(
    model, train_loader, val_loader,
    criterion_p2, optimizer_p2, scheduler_p2,
    NUM_EPOCHS_FINETUNE,
    "Phase 2: Full Fine-tune (all layers)"
)

In [ ]:
# ── 14. Training curves plot ────────────────────────────────────────────────────

def plot_history(h1, h2, save_path="/kaggle/working/training_curves.png"):
    epochs_p1 = list(range(1, len(h1["train_acc"]) + 1))
    epochs_p2 = list(range(len(h1["train_acc"]) + 1,
                           len(h1["train_acc"]) + len(h2["train_acc"]) + 1))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    axes[0].plot(epochs_p1, h1["train_acc"], "b-o",  label="P1 train", markersize=4)
    axes[0].plot(epochs_p1, h1["val_acc"],   "b--s", label="P1 val",   markersize=4)
    axes[0].plot(epochs_p2, h2["train_acc"], "r-o",  label="P2 train", markersize=4)
    axes[0].plot(epochs_p2, h2["val_acc"],   "r--s", label="P2 val",   markersize=4)
    axes[0].axvline(x=len(h1["train_acc"]) + 0.5, color="gray", linestyle=":", label="fine-tune start")
    axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()

    # Loss
    axes[1].plot(epochs_p1, h1["train_loss"], "b-o",  markersize=4)
    axes[1].plot(epochs_p1, h1["val_loss"],   "b--s", markersize=4)
    axes[1].plot(epochs_p2, h2["train_loss"], "r-o",  markersize=4)
    axes[1].plot(epochs_p2, h2["val_loss"],   "r--s", markersize=4)
    axes[1].axvline(x=len(h1["train_loss"]) + 0.5, color="gray", linestyle=":")
    axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch")

    plt.suptitle(f"Training Curves — {MODEL_NAME}")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Training curves saved: {save_path}")

plot_history(history_p1, history_p2)

## Evaluation — Test Set

In [ ]:
# ── 15. Test set evaluation ─────────────────────────────────────────────────────

criterion_eval = nn.CrossEntropyLoss()
test_loss, test_acc = evaluate(model, test_loader, criterion_eval)
print(f"Test Loss : {test_loss:.4f}")
print(f"Test Acc  : {test_acc:.4f}  ({test_acc * 100:.2f}%)")

In [ ]:
# ── 16. Collect predictions ─────────────────────────────────────────────────────

def collect_predictions(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images  = images.to(DEVICE)
            outputs = model(images)
            probs   = torch.softmax(outputs, dim=1).cpu().numpy()
            preds   = outputs.argmax(1).cpu().numpy()
            all_labels.extend(labels.numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


labels, preds, probs = collect_predictions(model, test_loader)
print(f"Collected {len(labels)} test samples.")

In [ ]:
# ── 17. Confusion Matrix ────────────────────────────────────────────────────────

def plot_confusion_matrix(labels, preds, class_names,
                          save_path="/kaggle/working/confusion_matrix.png"):
    cm  = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(len(class_names) + 2, len(class_names) + 2))
    im  = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    fig.colorbar(im, ax=ax)
    tick_marks = np.arange(len(class_names))
    ax.set_xticks(tick_marks); ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticks(tick_marks); ax.set_yticklabels(class_names)
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], "d"),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    ax.set_ylabel("True Label"); ax.set_xlabel("Predicted Label")
    ax.set_title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Confusion matrix saved: {save_path}")

plot_confusion_matrix(labels, preds, CLASS_NAMES)

In [ ]:
# ── 18. ROC Curve ────────────────────────────────────────────────────────────────

def plot_roc_curve(labels, probs, class_names,
                   save_path="/kaggle/working/roc_curve.png"):
    n_classes  = len(class_names)
    labels_bin = label_binarize(labels, classes=list(range(n_classes)))

    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(labels_bin[:, i], probs[:, i])
        roc_auc[i]        = auc(fpr[i], tpr[i])

    fpr["micro"], tpr["micro"], _ = roc_curve(labels_bin.ravel(), probs.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    colors = cycle(plt.cm.tab10.colors)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr["micro"], tpr["micro"],
            label=f"micro-avg (AUC={roc_auc['micro']:.3f})",
            color="red", linestyle=":", linewidth=2)
    for i, (cls, color) in enumerate(zip(class_names, colors)):
        ax.plot(fpr[i], tpr[i], color=color,
                label=f"{cls} (AUC={roc_auc[i]:.3f})")
    ax.plot([0, 1], [0, 1], "k--")
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve")
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"ROC curve saved: {save_path}")

plot_roc_curve(labels, probs, CLASS_NAMES)

In [ ]:
# ── 19. Classification Report ────────────────────────────────────────────────────

report = classification_report(labels, preds, target_names=CLASS_NAMES)
print("Classification Report:")
print(report)

report_path = Path("/kaggle/working/classification_report.txt")
report_path.write_text(report, encoding="utf-8")
print(f"Report saved: {report_path}")

## Save Model Weights & Full Model

In [ ]:
# ── 20. Save weights + full model ───────────────────────────────────────────────

out_dir = SAVE_BASE / MODEL_NAME
out_dir.mkdir(parents=True, exist_ok=True)

# Weights only
weights_path = out_dir / f"weights_{MODEL_NAME}.pth"
torch.save(model.state_dict(), weights_path)
print(f"Weights saved   : {weights_path}  "
      f"({weights_path.stat().st_size / 1024 / 1024:.1f} MB)")

# Full model (structure + weights)
full_path = out_dir / f"full_{MODEL_NAME}.pth"
torch.save(model, full_path)
print(f"Full model saved: {full_path}  "
      f"({full_path.stat().st_size / 1024 / 1024:.1f} MB)")

print(f"\nAll outputs → {out_dir}")

In [ ]:
# ── 21. List all output files ────────────────────────────────────────────────────

import glob
output_files = sorted(glob.glob("/kaggle/working/**", recursive=True))
print("Files in /kaggle/working/:")
for f in output_files:
    p = Path(f)
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  {f}  ({size_kb:.1f} KB)" if size_kb < 1024
              else f"  {f}  ({size_kb/1024:.1f} MB)")